### Deps

```shell
pip install oracledb langchain-oracledb langchain-core

```

In [1]:
import os
import oracledb
from langchain_core.documents import Document
from langchain_oracledb import OracleVS
from langchain_oracledb.embeddings import OracleEmbeddings
from langchain_oracledb.vectorstores import DistanceStrategy
from langchain_oracledb.vectorstores import oraclevs
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)

In [10]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

In [12]:
documents = [
    Document(
        page_content="""
        Oracle Database provides native vector search capabilities.
        HNSW is an approximate nearest neighbor algorithm used
        for high-performance semantic search.
        """,
        metadata={
            "source": "oracle_vector_search.txt"
        }
    )
]

In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 1


In [16]:
vector_store = OracleVS.from_documents(
    chunks,
    embedding_model,
    client=connection,
    table_name="DOCUMENT_VECTORS",
    distance_strategy=DistanceStrategy.COSINE
)

In [19]:
oraclevs.create_index(
    connection,
    vector_store,
    params={
        "idx_name": "DOCUMENT_HNSW_IDX",
        "idx_type": "HNSW",
        "neighbors": 32,
        "efConstruction": 200,
        "accuracy": 95
    }
)

print("HNSW index created.")

HNSW index created.


In [20]:
query = "How does Oracle perform semantic vector search?"

results = vector_store.similarity_search(
    query,
    k=5
)

In [21]:
for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Content:", result.page_content)
    print("Metadata:", result.metadata)


Result 1
Content: Oracle Database provides native vector search capabilities.
        HNSW is an approximate nearest neighbor algorithm used
        for high-performance semantic search.
Metadata: {'source': 'oracle_vector_search.txt'}


In [22]:
documents2 = [
    Document(
        page_content="""
        OpenSearch features a vector database engine capable of handling 
k-NN similarity search, making it an excellent backend choice for 
local, private RAG workflows powered by Ollama.
        """,
        metadata={
            "source": "opensearch_vector_search.txt"
        }
    )
]

In [23]:
chunks2 = splitter.split_documents(documents2)

print("Number of chunks:", len(chunks2))

Number of chunks: 1


In [24]:
# Insert second document into the existing Oracle vector store
vector_store.add_documents(chunks2)

print("Second document inserted.")

Second document inserted.


In [25]:
query = "What can be used as a vector database for private RAG?"

results = vector_store.similarity_search(
    query,
    k=5
)

In [26]:
for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print(result.page_content)
    print(result.metadata)


Result 1
OpenSearch features a vector database engine capable of handling 
k-NN similarity search, making it an excellent backend choice for 
local, private RAG workflows powered by Ollama.
{'source': 'opensearch_vector_search.txt'}

Result 2
Oracle Database provides native vector search capabilities.
        HNSW is an approximate nearest neighbor algorithm used
        for high-performance semantic search.
{'source': 'oracle_vector_search.txt'}


In [27]:
query = "What is OpenSearch?"

results = vector_store.similarity_search(
    query,
    k=1
)
for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print(result.page_content)
    print(result.metadata)


Result 1
OpenSearch features a vector database engine capable of handling 
k-NN similarity search, making it an excellent backend choice for 
local, private RAG workflows powered by Ollama.
{'source': 'opensearch_vector_search.txt'}
